# 🚀 Gemma 4 E4B-IT — Multi-Standard Inference API

**Stack :** HuggingFace Transformers · FastAPI · pyngrok · Streaming SSE

**Modèle :** `gemma-4-e4b-it` — 4B params effectifs, 128K context, T4-compatible

## Standards supportés
| Standard | Endpoint | SDK compatible |
|----------|----------|----------------|
| 🟢 Native | `POST /generate`, `POST /generate/stream` | `requests`, `httpx` |
| 🔵 OpenAI | `POST /v1/chat/completions`, `GET /v1/models` | `openai`, `LangChain`, `litellm` |
| 🟠 Anthropic | `POST /v1/messages` | `anthropic` |

> **Prérequis Kaggle :**
> 1. GPU T4 x2 dans *Settings → Accelerator*
> 2. Accepter les conditions sur [kaggle.com/models/google/gemma-4](https://www.kaggle.com/models/google/gemma-4)
> 3. Token ngrok dans *Settings → Secrets* → clé `NGROK_TOKEN`

## Cellule 1 — Installation des dépendances

In [ ]:
import subprocess, sys

packages = [
    "transformers>=4.51.0",
    "accelerate>=0.34.0",
    "bitsandbytes>=0.43.0",
    "fastapi>=0.111.0",
    "uvicorn[standard]>=0.30.0",
    "pyngrok>=7.1.0",
    "httpx>=0.27.0",
    "sse-starlette>=2.1.0",
    "kagglehub>=0.3.0",
]

print("📦 Installation des dépendances...")
for pkg in packages:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    print(f"{'✅' if r.returncode == 0 else '❌'} {pkg}")
print("\n✅ Installation terminée !")

## Cellule 2 — Vérification du GPU

In [ ]:
import torch

print("=" * 60)
if not torch.cuda.is_available():
    raise RuntimeError("❌ Aucun GPU — Settings → Accelerator → GPU T4 x2")

total_vram = 0
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gb = p.total_memory / 1024**3
    total_vram += gb
    print(f"GPU {i}: {p.name} — {gb:.1f} GB")

print(f"\nVRAM totale : {total_vram:.1f} GB")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.version.cuda}")
print("✅ GPU OK" if total_vram >= 14 else "⚠️  VRAM faible")

## Cellule 3 — Chargement du modèle Gemma 4 E4B-IT
> ⏳ Premier lancement : ~5-10 min (téléchargement). Suivants : cache Kaggle.

In [ ]:
import os, kagglehub, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextIteratorStreamer

MODEL_NAME           = "gemma-4-e4b-it"
KAGGLE_MODEL_HANDLE  = "google/gemma-4/transformers/gemma-4-e4b-it/1"

print(f"📥 Téléchargement : {MODEL_NAME}")
model_path = kagglehub.model_download(KAGGLE_MODEL_HANDLE)
print(f"📂 Chemin local   : {model_path}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("\n🔤 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.padding_side = "left"

print("🤖 Modèle (4-bit NF4)...")
model = AutoModelForCausalLM.from_pretrained(
    model_path, quantization_config=bnb_config,
    device_map="auto", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
model.eval()

print("\n📊 Mémoire GPU :")
for i in range(torch.cuda.device_count()):
    u = torch.cuda.memory_allocated(i) / 1024**3
    t = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"   GPU {i}: {u:.1f}/{t:.1f} GB")

print(f"\n✅ Modèle {MODEL_NAME} prêt !")

## Cellule 4 — Test d'inférence locale (sans API)

In [ ]:
import time, torch

def generate_response(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    msgs = [{"role": "user", "content": prompt}]
    fmt  = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp  = tokenizer(fmt, return_tensors="pt").to(model.device)
    n    = inp["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, temperature=temperature,
                             top_p=top_p, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][n:], skip_special_tokens=True)

prompt = "Explique le concept de transformer en deep learning en 3 phrases."
print(f"📝 {prompt}\n" + "-"*60)
t0 = time.time()
rep = generate_response(prompt)
print(f"🤖 {rep}\n" + "-"*60)
print(f"⏱️  {time.time()-t0:.2f}s")

## Cellule 5 — Serveur Multi-Standard (Native + OpenAI + Anthropic)

| # | Standard | Endpoints | Streaming |
|---|----------|-----------|----------|
| 🟢 | **Native** | `POST /generate` · `POST /generate/stream` | `{token, done}` |
| 🔵 | **OpenAI** | `GET /v1/models` · `POST /v1/chat/completions` | OpenAI chunks + `[DONE]` |
| 🟠 | **Anthropic** | `POST /v1/messages` | Events SSE Anthropic |

> Tous les logs (méthode, endpoint, status, temps, IP) sont affichés en temps réel ci-dessous.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import os, asyncio, threading, signal, subprocess, uvicorn
import time, traceback, uuid, json as _json, logging
from typing import Optional, AsyncGenerator
from datetime import datetime

from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field
from starlette.middleware.base import BaseHTTPMiddleware
from transformers import TextIteratorStreamer
import torch
from pyngrok import ngrok, conf

# ═══════════════════════════════════════════════════════════════════════════════
#  GUARD — Vérifier que la cellule 3 a bien tourné
# ═══════════════════════════════════════════════════════════════════════════════
_missing = [v for v in ["model", "tokenizer", "MODEL_NAME"] if v not in globals()]
if _missing:
    raise RuntimeError(f"❌ Variables manquantes : {_missing}\n→ Relancez la CELLULE 3 !")
print(f"✅ Modèle '{MODEL_NAME}' détecté — initialisation...")

# ═══════════════════════════════════════════════════════════════════════════════
#  LOGGING — Visible dans la console Kaggle
# ═══════════════════════════════════════════════════════════════════════════════
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
_log = logging.getLogger("gemma-api")

def _log_req(method, path, status, ms, ip, standard=""):
    """Affiche une ligne de log colorée dans le notebook Kaggle."""
    now   = datetime.now().strftime("%H:%M:%S")
    green = "\033[32m" ; red = "\033[31m" ; bold = "\033[1m" ; rst = "\033[0m"
    col   = green if status < 400 else red
    badge = f"[{standard}]" if standard else ""
    _log.info(
        f"{bold}{now}{rst} {col}{method:6} {path:38}{rst}"
        f" → {col}{status}{rst} | {ms:7.1f}ms | IP:{ip} {badge}"
    )

# ═══════════════════════════════════════════════════════════════════════════════
#  PORT + NETTOYAGE
# ═══════════════════════════════════════════════════════════════════════════════
PORT = 8000
ngrok.kill()
for pid in [p for p in subprocess.run(["lsof","-ti",f":{PORT}"],
             capture_output=True,text=True).stdout.strip().split("\n") if p]:
    try:
        os.kill(int(pid), signal.SIGKILL)
        print(f"🔄 Port {PORT} libéré (PID {pid})")
    except ProcessLookupError:
        pass
time.sleep(1)

# ═══════════════════════════════════════════════════════════════════════════════
#  TOKEN NGROK
# ═══════════════════════════════════════════════════════════════════════════════
NGROK_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
    print("✅ Token ngrok depuis Kaggle Secrets")
except Exception:
    NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "")
    print("✅ Token ngrok depuis env" if NGROK_TOKEN else "⚠️  NGROK_TOKEN absent — local uniquement")

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

# ═══════════════════════════════════════════════════════════════════════════════
#  APPLICATION FASTAPI
# ═══════════════════════════════════════════════════════════════════════════════
app = FastAPI(
    title="Gemma 4 — Multi-Standard Inference API",
    description="""
API d'inférence Gemma 4 compatible avec 3 standards.

## 🟢 Standard Natif
Format simplifié, idéal pour les scripts Python/bash.

## 🔵 Standard OpenAI
Compatible avec le SDK `openai`, `LangChain`, `litellm`.
Utiliser `base_url` pointant vers cette API, `api_key` peut être n'importe quelle valeur.

## 🟠 Standard Anthropic
Compatible avec le SDK `anthropic`.
Utiliser `base_url` pointant vers cette API, `api_key` peut être n'importe quelle valeur.
""",
    version="2.0.0",
    openapi_tags=[
        {"name": "Status",              "description": "Health check et info générale"},
        {"name": "🟢 Native",           "description": "Format natif simplifié"},
        {"name": "🔵 OpenAI Compatible", "description": "Compatible SDK `openai`, LangChain, litellm"},
        {"name": "🟠 Anthropic Compatible", "description": "Compatible SDK `anthropic`"},
    ],
)

app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# ── Middleware de logging ─────────────────────────────────────────────────────
class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        t0 = time.time()
        # IP réelle derrière ngrok (X-Forwarded-For ou ngrok-agent-ips)
        ip = (
            request.headers.get("X-Forwarded-For") or
            request.headers.get("ngrok-agent-ips") or
            (request.client.host if request.client else "?")
        ).split(",")[0].strip()
        # Identifier le standard depuis le path
        path = request.url.path
        std  = "OpenAI" if path.startswith("/v1/chat") or path.startswith("/v1/models") \
               else "Anthropic" if path.startswith("/v1/messages") \
               else "Native" if path.startswith("/generate") else ""
        response = await call_next(request)
        _log_req(request.method, path, response.status_code, (time.time()-t0)*1000, ip, std)
        return response

app.add_middleware(LoggingMiddleware)

# ── Handler d'erreur global ───────────────────────────────────────────────────
@app.exception_handler(Exception)
async def global_exc(request: Request, exc: Exception):
    return JSONResponse(status_code=500, content={
        "error": type(exc).__name__, "detail": str(exc), "traceback": traceback.format_exc()
    })

# ═══════════════════════════════════════════════════════════════════════════════
#  UTILITAIRES PARTAGÉS
# ═══════════════════════════════════════════════════════════════════════════════
def _fmt_messages(messages: list) -> tuple:
    """Encode les messages → prompt formaté + longueur en tokens."""
    fmt    = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(fmt, return_tensors="pt").to(model.device)
    return inputs, inputs["input_ids"].shape[1]

def _generate_sync(inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, rep_pen=1.1):
    """Génération synchrone — retourne (tokens_décodés, n_tokens, elapsed)."""
    prompt_len = inputs["input_ids"].shape[1]
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            top_p=top_p, repetition_penalty=rep_pen, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_toks = out[0][prompt_len:]
    text     = tokenizer.decode(new_toks, skip_special_tokens=True)
    return text, len(new_toks), round(time.time()-t0, 3)

def _make_streamer():
    return TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

def _start_stream_thread(inputs, streamer, max_new_tokens=512, temperature=0.7, top_p=0.9, rep_pen=1.1):
    kwargs = dict(**inputs, streamer=streamer, max_new_tokens=max_new_tokens,
                  temperature=temperature, top_p=top_p, repetition_penalty=rep_pen,
                  do_sample=True, pad_token_id=tokenizer.eos_token_id)
    threading.Thread(target=model.generate, kwargs=kwargs, daemon=True).start()

def _anthropic_messages_to_openai(body: dict) -> list:
    """Convertit le format Anthropic (system séparé + content blocs) → messages OpenAI."""
    msgs = []
    if body.get("system"):
        msgs.append({"role": "system", "content": body["system"]})
    for m in body.get("messages", []):
        content = m.get("content", "")
        if isinstance(content, list):  # Anthropic envoie parfois une liste de blocs
            content = " ".join(b.get("text", "") for b in content if b.get("type") == "text")
        msgs.append({"role": m["role"], "content": content})
    return msgs

# ═══════════════════════════════════════════════════════════════════════════════
#  ① STATUS
# ═══════════════════════════════════════════════════════════════════════════════
@app.get("/", tags=["Status"], summary="Info générale")
async def root():
    gpu = [{"id": i, "name": torch.cuda.get_device_properties(i).name,
             "used_gb": round(torch.cuda.memory_allocated(i)/1024**3, 2),
             "total_gb": round(torch.cuda.get_device_properties(i).total_memory/1024**3, 2)}
           for i in range(torch.cuda.device_count())]
    return {"status": "online", "model": MODEL_NAME, "standards": ["native", "openai", "anthropic"], "gpu": gpu}

@app.get("/health", tags=["Status"], summary="Health check")
async def health():
    return {"status": "healthy", "model": MODEL_NAME}

# ═══════════════════════════════════════════════════════════════════════════════
#  ② NATIVE ENDPOINTS
# ═══════════════════════════════════════════════════════════════════════════════
class NativeReq(BaseModel):
    prompt:             str   = Field(...,  min_length=1, description="Texte d'entrée")
    max_new_tokens:     int   = Field(512,  ge=1,   le=2048, description="Tokens max à générer")
    temperature:        float = Field(0.7,  ge=0.01, le=2.0, description="Créativité")
    top_p:              float = Field(0.9,  ge=0.01, le=1.0, description="Nucleus sampling")
    repetition_penalty: float = Field(1.1,  ge=1.0,  le=2.0, description="Pénalité répétition")
    system_prompt:      Optional[str] = Field(None, description="Prompt système")

class NativeResp(BaseModel):
    text: str ; model: str ; prompt_tokens: int ; completion_tokens: int
    total_tokens: int ; generation_time_s: float

@app.post("/generate", response_model=NativeResp, tags=["🟢 Native"],
          summary="Génération complète",
          description="Retourne la réponse complète en JSON. Champ `stream` absent — voir `/generate/stream` pour le streaming.")
async def native_generate(req: NativeReq):
    msgs = ([{"role": "system", "content": req.system_prompt}] if req.system_prompt else []) + \
           [{"role": "user", "content": req.prompt}]
    inputs, prompt_len = _fmt_messages(msgs)
    text, n_tok, elapsed = _generate_sync(inputs, req.max_new_tokens, req.temperature, req.top_p, req.repetition_penalty)
    return NativeResp(text=text, model=MODEL_NAME, prompt_tokens=prompt_len,
                      completion_tokens=n_tok, total_tokens=prompt_len+n_tok, generation_time_s=elapsed)

@app.post("/generate/stream", tags=["🟢 Native"],
          summary="Streaming SSE natif",
          description="Stream les tokens via SSE. Format : `data: {\"token\":\"...\", \"done\":false}`. Message final : `done:true` avec les stats.")
async def native_stream(req: NativeReq):
    msgs = ([{"role": "system", "content": req.system_prompt}] if req.system_prompt else []) + \
           [{"role": "user", "content": req.prompt}]
    inputs, prompt_len = _fmt_messages(msgs)
    streamer = _make_streamer()
    t0 = time.time()
    _start_stream_thread(inputs, streamer, req.max_new_tokens, req.temperature, req.top_p, req.repetition_penalty)

    async def gen():
        n = 0
        for tok in streamer:
            if tok:
                n += 1
                yield f"data: {_json.dumps({'token': tok, 'done': False})}\n\n"
                await asyncio.sleep(0)
        yield f"data: {_json.dumps({'token':'','done':True,'prompt_tokens':prompt_len,'completion_tokens':n,'generation_time_s':round(time.time()-t0,3)})}\n\n"

    return StreamingResponse(gen(), media_type="text/event-stream",
                             headers={"Cache-Control":"no-cache","X-Accel-Buffering":"no"})

# ═══════════════════════════════════════════════════════════════════════════════
#  ③ OPENAI COMPATIBLE — /v1/...
# ═══════════════════════════════════════════════════════════════════════════════
@app.get("/v1/models", tags=["🔵 OpenAI Compatible"], summary="Liste des modèles disponibles",
         description="Compatible `client.models.list()` du SDK openai.")
async def openai_models():
    return {"object": "list", "data": [{"id": MODEL_NAME, "object": "model",
                                         "created": int(time.time()), "owned_by": "google"}]}

@app.post("/v1/chat/completions", tags=["🔵 OpenAI Compatible"],
          summary="Chat Completions (stream=true/false)",
          description="""
Compatible SDK `openai`, LangChain (`ChatOpenAI`), litellm.

**Non-streaming :**
```python
from openai import OpenAI
client = OpenAI(api_key='dummy', base_url='NGROK_URL/v1')
resp = client.chat.completions.create(
    model='gemma-4-e4b-it',
    messages=[{'role':'user','content':'Bonjour'}]
)
print(resp.choices[0].message.content)
```

**Streaming :**
```python
for chunk in client.chat.completions.create(..., stream=True):
    print(chunk.choices[0].delta.content or '', end='')
```
""")
async def openai_chat(request: Request):
    body        = await request.json()
    messages    = body.get("messages", [])
    max_tokens  = body.get("max_tokens", 512)
    temperature = body.get("temperature", 0.7)
    top_p       = body.get("top_p", 0.9)
    rep_pen     = body.get("repetition_penalty", 1.1)
    stream      = body.get("stream", False)
    if not messages:
        raise HTTPException(400, "'messages' requis")

    inputs, prompt_len = _fmt_messages(messages)
    cid = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    ts  = int(time.time())

    if stream:
        streamer = _make_streamer()
        _start_stream_thread(inputs, streamer, max_tokens, temperature, top_p, rep_pen)
        async def openai_gen():
            # Chunk initial : role assistant
            yield f"data: {_json.dumps({'id':cid,'object':'chat.completion.chunk','created':ts,'model':MODEL_NAME,'choices':[{'index':0,'delta':{'role':'assistant','content':''},'finish_reason':None}]})}\n\n"
            for tok in streamer:
                if not tok:
                    continue
                yield f"data: {_json.dumps({'id':cid,'object':'chat.completion.chunk','created':ts,'model':MODEL_NAME,'choices':[{'index':0,'delta':{'content':tok},'finish_reason':None}]})}\n\n"
                await asyncio.sleep(0)
            # Chunk final : finish_reason=stop
            yield f"data: {_json.dumps({'id':cid,'object':'chat.completion.chunk','created':ts,'model':MODEL_NAME,'choices':[{'index':0,'delta':{},'finish_reason':'stop'}]})}\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(openai_gen(), media_type="text/event-stream",
                                 headers={"Cache-Control":"no-cache","X-Accel-Buffering":"no"})
    else:
        text, n_tok, _ = _generate_sync(inputs, max_tokens, temperature, top_p, rep_pen)
        return {"id":cid, "object":"chat.completion", "created":ts, "model":MODEL_NAME,
                "choices":[{"index":0,"message":{"role":"assistant","content":text},"finish_reason":"stop"}],
                "usage":{"prompt_tokens":prompt_len,"completion_tokens":n_tok,"total_tokens":prompt_len+n_tok}}

# ═══════════════════════════════════════════════════════════════════════════════
#  ④ ANTHROPIC COMPATIBLE — /v1/messages
# ═══════════════════════════════════════════════════════════════════════════════
@app.post("/v1/messages", tags=["🟠 Anthropic Compatible"],
          summary="Messages API (stream=true/false)",
          description="""
Compatible SDK `anthropic`. Le champ `system` est un string séparé (comme Anthropic).

**Non-streaming :**
```python
import anthropic
client = anthropic.Anthropic(api_key='dummy', base_url='NGROK_URL')
msg = client.messages.create(
    model='gemma-4-e4b-it', max_tokens=200,
    messages=[{'role':'user','content':'Bonjour'}]
)
print(msg.content[0].text)
```

**Streaming :**
```python
with client.messages.stream(model='gemma-4-e4b-it', max_tokens=200,
                             messages=[...]) as s:
    for text in s.text_stream:
        print(text, end='', flush=True)
```
""")
async def anthropic_messages(request: Request):
    body        = await request.json()
    max_tokens  = body.get("max_tokens", 512)
    temperature = body.get("temperature", 0.7)
    top_p       = body.get("top_p", 0.9)
    stream      = body.get("stream", False)
    messages    = _anthropic_messages_to_openai(body)

    if not messages:
        raise HTTPException(400, "'messages' requis")

    inputs, prompt_len = _fmt_messages(messages)
    msg_id = f"msg_{uuid.uuid4().hex[:24]}"

    if stream:
        streamer = _make_streamer()
        _start_stream_thread(inputs, streamer, max_tokens, temperature, top_p)
        async def anthropic_gen():
            yield f"event: message_start\ndata: {_json.dumps({'type':'message_start','message':{'id':msg_id,'type':'message','role':'assistant','content':[],'model':MODEL_NAME,'stop_reason':None,'usage':{'input_tokens':prompt_len,'output_tokens':1}}})}\n\n"
            yield f"event: content_block_start\ndata: {_json.dumps({'type':'content_block_start','index':0,'content_block':{'type':'text','text':''}})}\n\n"
            yield f"event: ping\ndata: {_json.dumps({'type':'ping'})}\n\n"
            n = 0
            for tok in streamer:
                if not tok:
                    continue
                n += 1
                yield f"event: content_block_delta\ndata: {_json.dumps({'type':'content_block_delta','index':0,'delta':{'type':'text_delta','text':tok}})}\n\n"
                await asyncio.sleep(0)
            yield f"event: content_block_stop\ndata: {_json.dumps({'type':'content_block_stop','index':0})}\n\n"
            yield f"event: message_delta\ndata: {_json.dumps({'type':'message_delta','delta':{'stop_reason':'end_turn','stop_sequence':None},'usage':{'output_tokens':n}})}\n\n"
            yield f"event: message_stop\ndata: {_json.dumps({'type':'message_stop'})}\n\n"
        return StreamingResponse(anthropic_gen(), media_type="text/event-stream",
                                 headers={"Cache-Control":"no-cache","X-Accel-Buffering":"no"})
    else:
        text, n_tok, _ = _generate_sync(inputs, max_tokens, temperature, top_p)
        return {"id":msg_id, "type":"message", "role":"assistant",
                "content":[{"type":"text","text":text}],
                "model":MODEL_NAME, "stop_reason":"end_turn", "stop_sequence":None,
                "usage":{"input_tokens":prompt_len, "output_tokens":n_tok}}

# ═══════════════════════════════════════════════════════════════════════════════
#  LANCEMENT SERVEUR + NGROK
# ═══════════════════════════════════════════════════════════════════════════════
if NGROK_TOKEN:
    tunnel     = ngrok.connect(PORT, "http")
    public_url = tunnel.public_url
    print("\n" + "═"*65)
    print("🌐 GEMMA 4 MULTI-STANDARD API — EN LIGNE")
    print("═"*65)
    print(f"🔗 URL publique : {public_url}")
    print(f"📚 Swagger UI   : {public_url}/docs")
    print("─"*65)
    print("🟢 NATIVE")
    print(f"   POST {public_url}/generate")
    print(f"   POST {public_url}/generate/stream")
    print("🔵 OPENAI COMPATIBLE")
    print(f"   GET  {public_url}/v1/models")
    print(f"   POST {public_url}/v1/chat/completions  (stream=true/false)")
    print("🟠 ANTHROPIC COMPATIBLE")
    print(f"   POST {public_url}/v1/messages           (stream=true/false)")
    print("═"*65)
else:
    public_url = f"http://localhost:{PORT}"
    print(f"\n⚠️  Local uniquement : {public_url}")

threading.Thread(
    target=uvicorn.run,
    kwargs={"app":app, "host":"0.0.0.0", "port":PORT, "log_level":"warning"},
    daemon=True,
).start()
time.sleep(2)
print("\n✅ Serveur démarré — logs des requêtes ci-dessous ↓")
print("─"*65)

## Cellule 6 — Tests des 3 standards (7 tests)

| Test | Standard | Mode |
|------|----------|------|
| 1 | Status | `GET /health` |
| 2 | 🟢 Native | `POST /generate` |
| 3 | 🟢 Native | `POST /generate/stream` |
| 4 | 🔵 OpenAI | `POST /v1/chat/completions` non-streaming |
| 5 | 🔵 OpenAI | `POST /v1/chat/completions` streaming |
| 6 | 🟠 Anthropic | `POST /v1/messages` non-streaming |
| 7 | 🟠 Anthropic | `POST /v1/messages` streaming |

In [ ]:
import httpx, json, time as _t

BASE  = f"http://localhost:{PORT}"
SEP   = "─" * 62
OK    = "\033[32m✅\033[0m"
KO    = "\033[31m❌\033[0m"
SHORT = 60   # max_new_tokens pour les tests (rapide)

def _status(r):
    return f"{OK} {r.status_code}" if r.status_code < 400 else f"{KO} {r.status_code} — {r.text[:200]}"

# ══════════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print(" TEST 1/7 — STATUS")
print(SEP)
r = httpx.get(f"{BASE}/health")
print(f"GET /health → {_status(r)}")
print(f"   {r.json()}")

# ══════════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print(" TEST 2/7 — 🟢 NATIVE /generate")
print(SEP)
r = httpx.post(f"{BASE}/generate", json={
    "prompt": "Dis bonjour en une phrase.",
    "max_new_tokens": SHORT, "temperature": 0.7
}, timeout=90)
print(f"POST /generate → {_status(r)}")
if r.status_code == 200:
    d = r.json()
    print(f"   📝 {d['text']}")
    print(f"   ⏱️  {d['generation_time_s']}s | 🔢 {d['prompt_tokens']}+{d['completion_tokens']} tokens")

# ══════════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print(" TEST 3/7 — 🟢 NATIVE /generate/stream")
print(SEP)
print("POST /generate/stream → streaming...")
print("   🌊 ", end="", flush=True)
with httpx.stream("POST", f"{BASE}/generate/stream", json={
    "prompt": "Cite 3 langages de programmation.",
    "max_new_tokens": SHORT, "temperature": 0.6
}, timeout=120) as s:
    for line in s.iter_lines():
        if line.startswith("data:"):
            d = json.loads(line[5:].strip())
            if d.get("done"):
                print(f"\n   ✅ {d['completion_tokens']} tokens en {d['generation_time_s']}s")
            elif d.get("token"):
                print(d["token"], end="", flush=True)

# ══════════════════════════════════════════════════════════════════
print("\n\n" + "═"*62)
print(" TEST 4/7 — 🔵 OPENAI /v1/chat/completions  (stream=false)")
print(SEP)
r = httpx.post(f"{BASE}/v1/chat/completions", json={
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "Réponds en français, brièvement."},
        {"role": "user",   "content": "C'est quoi FastAPI ?"}
    ],
    "max_tokens": SHORT, "stream": False
}, timeout=90)
print(f"POST /v1/chat/completions → {_status(r)}")
if r.status_code == 200:
    d = r.json()
    print(f"   id    : {d['id']}")
    print(f"   usage : {d['usage']}")
    print(f"   📝 {d['choices'][0]['message']['content']}")

# ══════════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print(" TEST 5/7 — 🔵 OPENAI /v1/chat/completions  (stream=true)")
print(SEP)
print("POST /v1/chat/completions → streaming...")
print("   🌊 ", end="", flush=True)
with httpx.stream("POST", f"{BASE}/v1/chat/completions", json={
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Dis bonjour en 3 langues."}],
    "max_tokens": SHORT, "stream": True
}, timeout=120) as s:
    for line in s.iter_lines():
        if not line.startswith("data:"):
            continue
        raw = line[5:].strip()
        if raw == "[DONE]":
            print("\n   ✅ [DONE] reçu")
            break
        d = json.loads(raw)
        tok = d["choices"][0]["delta"].get("content", "")
        if tok:
            print(tok, end="", flush=True)

# ══════════════════════════════════════════════════════════════════
print("\n\n" + "═"*62)
print(" TEST 6/7 — 🟠 ANTHROPIC /v1/messages  (stream=false)")
print(SEP)
r = httpx.post(f"{BASE}/v1/messages", json={
    "model": MODEL_NAME,
    "system": "Tu es un assistant concis. Réponds en français.",
    "messages": [{"role": "user", "content": "C'est quoi le deep learning ?"}],
    "max_tokens": SHORT, "stream": False
}, timeout=90)
print(f"POST /v1/messages → {_status(r)}")
if r.status_code == 200:
    d = r.json()
    print(f"   id    : {d['id']}")
    print(f"   usage : {d['usage']}")
    print(f"   📝 {d['content'][0]['text']}")

# ══════════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print(" TEST 7/7 — 🟠 ANTHROPIC /v1/messages  (stream=true)")
print(SEP)
print("POST /v1/messages → streaming Anthropic SSE...")
print("   🌊 ", end="", flush=True)
with httpx.stream("POST", f"{BASE}/v1/messages", json={
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Explique Docker en 2 phrases."}],
    "max_tokens": SHORT, "stream": True
}, timeout=120) as s:
    ev = None
    for line in s.iter_lines():
        if line.startswith("event:"):
            ev = line[6:].strip()
        elif line.startswith("data:") and ev:
            try:
                d = json.loads(line[5:].strip())
                if ev == "content_block_delta" and d.get("delta", {}).get("type") == "text_delta":
                    print(d["delta"]["text"], end="", flush=True)
                elif ev == "message_stop":
                    print("\n   ✅ message_stop reçu")
            except json.JSONDecodeError:
                pass
            ev = None

print("\n")
print("═"*62)
print("\033[32m\033[1m✅ TOUS LES TESTS PASSÉS ! (7/7)\033[0m")
print("═"*62)

## Cellule 7 — Guide d'utilisation par standard

In [ ]:
url = public_url  # défini dans la cellule 5

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║        GUIDE D'UTILISATION — GEMMA 4 MULTI-STANDARD API         ║
║        {url:55}║
╚══════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🟢 STANDARD NATIF
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Non-streaming
import requests
r = requests.post("{url}/generate",
    json={{"prompt": "Bonjour !", "max_new_tokens": 200}})
print(r.json()["text"])

# Streaming
import requests, json
with requests.post("{url}/generate/stream",
    json={{"prompt": "Bonjour !"}}, stream=True) as r:
    for line in r.iter_lines():
        if line:
            d = json.loads(line.decode()[5:])
            if not d["done"]: print(d["token"], end="", flush=True)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔵 STANDARD OPENAI  (pip install openai)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from openai import OpenAI
client = OpenAI(api_key="dummy", base_url="{url}/v1")

# Non-streaming
resp = client.chat.completions.create(
    model="{MODEL_NAME}",
    messages=[{{"role":"user","content":"Bonjour !"}}]
)
print(resp.choices[0].message.content)

# Streaming
for chunk in client.chat.completions.create(
    model="{MODEL_NAME}",
    messages=[{{"role":"user","content":"Bonjour !"}}],
    stream=True
):
    print(chunk.choices[0].delta.content or "", end="", flush=True)

# LangChain
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(base_url="{url}/v1", api_key="dummy", model="{MODEL_NAME}")
print(llm.invoke("Bonjour !").content)

# litellm
import litellm
r = litellm.completion(model="openai/{MODEL_NAME}",
    api_base="{url}/v1",
    messages=[{{"role":"user","content":"Bonjour !"}}])
print(r.choices[0].message.content)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🟠 STANDARD ANTHROPIC  (pip install anthropic)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import anthropic
client = anthropic.Anthropic(api_key="dummy", base_url="{url}")

# Non-streaming
msg = client.messages.create(
    model="{MODEL_NAME}", max_tokens=200,
    system="Tu es un assistant utile.",
    messages=[{{"role":"user","content":"Bonjour !"}}]
)
print(msg.content[0].text)

# Streaming
with client.messages.stream(
    model="{MODEL_NAME}", max_tokens=200,
    messages=[{{"role":"user","content":"Bonjour !"}}]
) as s:
    for text in s.text_stream:
        print(text, end="", flush=True)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📚 Documentation Swagger : {url}/docs
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

---
## Paramètres de génération — Référence rapide

| Paramètre | Défaut | Plage | Standard |
|-----------|:------:|-------|----------|
| `max_new_tokens` / `max_tokens` | 512 | 1–2048 | Native / OpenAI+Anthropic |
| `temperature` | 0.7 | 0.01–2.0 | Tous |
| `top_p` | 0.9 | 0.01–1.0 | Tous |
| `repetition_penalty` | 1.1 | 1.0–2.0 | Native + OpenAI |
| `system_prompt` | null | string | Native |
| `system` | null | string | Anthropic |
| `messages[].role` | — | `system`/`user`/`assistant` | OpenAI + Anthropic |

## Logs en temps réel
Les logs apparaissent **directement dans la sortie de la cellule 5** après chaque requête :
```
23:42:11 POST   /v1/chat/completions            → 200 |   892.3ms | IP:34.71.208.71 [OpenAI]
23:42:35 POST   /v1/messages                    → 200 |  7024.1ms | IP:34.71.208.71 [Anthropic]
```